# 📡 Channel Model Governance Lab (LLM-Powered)

This notebook implements the **Quantitative 'Research Loop'** specifically for **Section II-B (Channel & Propagation Models)**.
It mirrors the logic of the *Quantitative Measure Governance Lab* but focuses on detecting and validating channel model usage using **Groq LLM**.

## Research Objectives (Section II-B)
1. **Detection**: Identify usage of specific channel models (Atmospheric, Turbulence, Pointing, Noise).
2. **Governance Check**: Validate symbol usage (e.g., is $h_{atm}$ defined via Beer-Lambert? Is $C_n^2$ usages consistent?)
3. **Evidence Extraction**: Extract precise quotes and locators for the manuscript.

## Target Scope
- **Atmospheric Loss**: Beer-Lambert, Extinction Coefficient, Fog/Rain params.
- **Turbulence**: Gamma-Gamma, Log-Normal, Málaga, Negative Exponential.
- **Pointing Errors**: Gaussian Beam Wander, Jitter, Zero-Boresight.
- **Noise Models**: Shot Noise dominant vs Thermal Noise dominant.


In [10]:
# @title 1. Install & Setup
!pip install -q groq

from google.colab import drive
import os
import glob
import json
import csv
from groq import Groq
from google.colab import userdata

# 1.1 Mount Drive & Set Base Dir
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST"

if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print(f"✅ Working Directory set to: {os.getcwd()}")
else:
    print(f"❌ Path not found: {BASE_DIR}. Please check your Drive structure.")

# 1.2 Load API Key
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    print("🔑 Groq API Key loaded.")
except Exception as e:
    print(f"⚠️ Error: {e}. Ensure 'GROQ_API_KEY' is in Colab Secrets.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Working Directory set to: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST
🔑 Groq API Key loaded.


In [11]:
# @title 2. Data Loader (Recursive)
def load_processed_markdowns(target_ids=None, limit=None):
    """
    Loads markdown files recursively from 'data/proc_markdowns'.
    Matches the logic of Quantitative_Measure_Governance_Lab.ipynb.
    """
    search_path = os.path.join("data", "processed_markdowns")

    # Recursive search for all .md files
    all_files = glob.glob(os.path.join(search_path, "**", "*.md"), recursive=True)

    # Filter for O_ISAC or COMST files
    valid_files = [f for f in all_files if "O_ISAC" in f or "COMST" in f]

    # ID Filtering
    selected_files = []
    if target_ids:
        print(f"Applying filter for {len(target_ids)} Target IDs...")
        for f_path in valid_files:
            p_id = os.path.basename(f_path).replace('.md', '')
            if p_id in target_ids:
                selected_files.append(f_path)
    else:
        selected_files = valid_files

    if limit:
        selected_files = selected_files[:limit]

    print(f"Found {len(valid_files)} total files. Loading {len(selected_files)} for analysis.")

    data = []
    for f_path in selected_files:
        p_id = os.path.basename(f_path).replace('.md', '')
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                content = f.read()
                data.append((p_id, content))
        except Exception as e:
            print(f"Error reading {f_path}: {e}")

    return data

In [12]:
# @title 3. Define II-B Research Agent (Enhanced Reasoning)
def analyze_channel_governance(paper_text, paper_id):
    """
    Asks LLM to extract Channel/Propagation model details from the paper.
    Now includes explicit 'Reasoning/Chain of Thought' step.
    """

    system_prompt = """
    You are a Senior Metrology Auditor (O-ISAC Survey Team).
    Your task is to identify exactly which **Channel Probability Distributions** and **Path Loss Models** are adopted in the paper.

    # CRITICAL: DISTINGUISH 'REVIEW' vs 'USAGE'
    - Many papers list models in their Intro/Related Work. IGNORE THESE.
    - You must find the **System Model** or **Channel Model** section where they define their equations.

    # TARGET CATEGORIES:
    1. **Atmospheric**: Beer-Lambert types, Visibility-based attenuation.
    2. **Turbulence**: Log-Normal (Weak), Gamma-Gamma (Moderate-Strong), Málaga (M), Negative Exponential.
    3. **Pointing**: Zero-Boresight, Gaussian Beam Wander, Beckmann.
    4. **Multipath/Indoor**: Ceiling bounce, Lambertian, Exponential Decay.

    # OUTPUT FORMAT:
    Return a JSON object with a 'reasoning' string and a 'findings' list.
    Example:
    {
      "reasoning": "I checked Section II. The authors explicitly define h_a using Beer-Lambert (Eq 3) and h_t using Gamma-Gamma (Eq 5). They mention Log-Normal only in the intro comparisons.",
      "findings": [
         { "category": "Turbulence", "model": "Gamma-Gamma", "status": "ADOPTED", "evidence": "h_t follows the Gamma-Gamma distribution..." },
         { "category": "Atmospheric", "model": "Beer-Lambert", "status": "ADOPTED", "evidence": "attenuation is given by h_l = exp(-sigma * L)" }
      ]
    }
    """

    user_prompt = f"""
    Paper ID: {paper_id}

    Analyze this text strictly. DO NOT hallucinate. If no specific channel model is defined (e.g. only experimental data), say 'NONE detected'.

    Text Content (First 35k chars):
    {paper_text[:35000]}
    """

    try:
        completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            # Using largest model for reasoning capability
            model="llama-3.3-70b-versatile",
            response_format={"type": "json_object"},
            temperature=0
        )
        return json.loads(completion.choices[0].message.content)
    except Exception as e:
        return {"error": str(e), "reasoning": "Fail", "findings": []}


In [15]:
# @title 4. Execute Research Loop (Targeted)

# ==========================================
# CONFIGURATION
# ==========================================
# Using the papers identified via Grep to ensure hits
TARGET_PAPERS = None
LIMIT = None
OUTPUT_CSV = "analysis/II_ev_v2/section2B_evidence_LLM.csv"
# ==========================================

# 1. Load Papers
papers = load_processed_markdowns(target_ids=TARGET_PAPERS, limit=LIMIT)

# 2. Run Agent
all_findings = []
print(f"\n🚀 Starting Refined II-B Analysis on {len(papers)} papers...\n")

for pid, text in papers:
    print(f"Processing {pid}...")
    result = analyze_channel_governance(text, pid)

    reasoning = result.get("reasoning", "No reasoning provided.")
    findings = result.get("findings", [])

    print(f"  🧠 Agent Reasoning: {reasoning[:150]}...")

    for f in findings:
        if f.get('status') == 'ADOPTED':
            f["paper_id"] = pid
            f["full_reasoning"] = reasoning
            all_findings.append(f)
            print(f"     -> ✅ Found {f.get('category')}: {f.get('model')}")
    print("-"*40)

# 3. Export Results
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
if all_findings:
    keys = ["paper_id", "category", "model", "status", "evidence", "full_reasoning"]
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_findings)
    print(f"\n✅ Saved {len(all_findings)} confirmed evidence rows to {OUTPUT_CSV}")
    # Show first few results
    import pandas as pd
    display(pd.read_csv(OUTPUT_CSV).head())
else:
    print("\n⚠️ No findings extracted.")

Found 313 total files. Loading 313 for analysis.

🚀 Starting Refined II-B Analysis on 313 papers...

Processing O_ISAC_029...
  🧠 Agent Reasoning: The paper does not explicitly define channel models or probability distributions in the provided text. The focus is on the system design and experimen...
----------------------------------------
Processing O_ISAC_029...
  🧠 Agent Reasoning: I checked Section II. The authors explicitly define the system model using equations but do not mention specific channel models such as Beer-Lambert, ...
----------------------------------------
Processing O_ISAC_001...
  🧠 Agent Reasoning: The paper does not explicitly define a channel model or probability distribution in the provided text. The focus is on modulation strategies for optic...
----------------------------------------
Processing O_ISAC_001...
  🧠 Agent Reasoning: The paper does not explicitly define a channel model or probability distribution in the provided text. The focus is on modulation 

,paper_id,category,model,status,evidence,full_reasoning
0,O_ISAC_005,Atmospheric,Beer-Lambert,ADOPTED,The attenuation coefficient vartheta is determ...,I checked Section II where the system model is...
1,O_ISAC_005,Atmospheric,Beer-Lambert,ADOPTED,The attenuation coefficient $\vartheta$ is det...,I checked Section II where the authors define ...
2,O_ISAC_022,Multipath/Indoor,Lambertian,ADOPTED,The order of Lambertian emission of the i-th l...,I checked Section II where the authors define ...
3,O_ISAC_022,Multipath/Indoor,Line-Of-Sight (LOS),ADOPTED,"The NLOS component is not taken into account, ...",I checked Section II where the authors define ...
4,O_ISAC_022,Multipath/Indoor,Line-of-Sight (LOS),ADOPTED,The channel impulse response is composed of a ...,I checked Section II where the authors define ...
